# 组合优化与帕累托分析

本 notebook 分析 FlashAttention 各版本性能对比及多优化组合的帕累托前沿，对应论文：
- **Fig.4a**: FA 版本对比柱状图
- **Fig.4b**: 加速比随序列长度变化折线图
- **Fig.4c**: FA 对内存的影响
- **Fig.4d**: 小模型 vs 大模型 FA 收益对比
- **Fig.6a**: 加速比-精度帕累托前沿散点图
- **Fig.6b**: 最优组合雷达图

In [ ]:
import sys
import json
import subprocess
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

# 项目路径
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# 结果目录
FA_RESULTS_DIR = PROJECT_ROOT / "results" / "flash_attention"
COMBINED_RESULTS_DIR = PROJECT_ROOT / "results" / "combined"
FA_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
COMBINED_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"项目根目录: {PROJECT_ROOT}")
print("导入完成")

In [ ]:
# 运行 FlashAttention 实验（或加载已有结果）
fa_results_file = FA_RESULTS_DIR / "fa_results_for_notebook.json"

if fa_results_file.exists():
    with open(fa_results_file, 'r') as f:
        fa_results = json.load(f)
    print(f"已加载 FA 实验结果: {len(fa_results)} 条记录")
else:
    print("运行 FlashAttention 实验...")
    result = subprocess.run(
        [sys.executable, str(PROJECT_ROOT / "scripts" / "flash_attention_experiment.py"),
         "--models", "scgpt", "geneformer", "scfoundation",
         "--attention-versions", "native", "fa1", "fa2", "fa3", "sdpa",
         "--sequence-lengths", "128", "256", "512", "1024", "2048", "4096",
         "--output", str(FA_RESULTS_DIR)],
        capture_output=True, text=True, cwd=str(PROJECT_ROOT)
    )
    print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr[-1000:])
    # 加载最新结果
    result_files = sorted(FA_RESULTS_DIR.glob("flash_attention_results_*.json"))
    if result_files:
        with open(result_files[-1], 'r') as f:
            fa_results = json.load(f)
        # 缓存
        with open(fa_results_file, 'w') as f:
            json.dump(fa_results, f)
        print(f"FA 实验完成，共 {len(fa_results)} 条记录")
    else:
        print("未找到结果文件，使用模拟数据")
        fa_results = []

In [ ]:
# Fig.4a: FA 版本对比柱状图
def plot_fig4a(results):
    """FA 版本对比柱状图 - 对应论文 Fig.4a"""
    version_data = [r for r in results if r.get('sequence_length') == 512
                    and r.get('experiment') not in ('custom_mask', 'sequence_length_sweep', 'model_size_comparison')]
    if not version_data:
        print("无版本对比数据，跳过")
        return
    models = sorted(set(r.get('model', '') for r in version_data))
    versions = ['native', 'fa1', 'fa2', 'fa3', 'sdpa']
    x = np.arange(len(versions))
    width = 0.25
    fig, ax = plt.subplots(figsize=(12, 6))
    colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336', '#9C27B0']
    for i, model in enumerate(models):
        model_data = [r for r in version_data if r.get('model') == model]
        speedups = []
        for v in versions:
            match = [r for r in model_data if r.get('attention_version') == v]
            speedups.append(match[0].get('speedup_vs_native', 1.0) if match else 1.0)
        bars = ax.bar(x + i * width, speedups, width, label=model, color=colors[i % len(colors)], alpha=0.85)
        for bar, val in zip(bars, speedups):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                    f'{val:.2f}x', ha='center', va='bottom', fontsize=9)
    ax.set_xlabel('Attention Version', fontsize=12)
    ax.set_ylabel('Speedup vs Native', fontsize=12)
    ax.set_title('Fig.4a: FlashAttention Version Comparison (seq_len=512)', fontsize=14, fontweight='bold')
    ax.set_xticks(x + width)
    ax.set_xticklabels(versions)
    ax.legend()
    ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(FA_RESULTS_DIR / 'fig4a_fa_version_comparison.png', dpi=150)
    plt.show()

plot_fig4a(fa_results)

In [ ]:
# Fig.4b: 加速比随序列长度变化折线图
def plot_fig4b(results):
    """加速比随序列长度变化 - 对应论文 Fig.4b"""
    sweep_data = [r for r in results if r.get('experiment') == 'sequence_length_sweep']
    if not sweep_data:
        print("无序列长度扫描数据，跳过")
        return
    versions = ['fa1', 'fa2', 'fa3', 'sdpa']
    colors = {'fa1': '#4CAF50', 'fa2': '#FF9800', 'fa3': '#F44336', 'sdpa': '#9C27B0'}
    markers = {'fa1': 'o', 'fa2': 's', 'fa3': '^', 'sdpa': 'D'}
    fig, ax = plt.subplots(figsize=(10, 6))
    for v in versions:
        v_data = [r for r in sweep_data if r.get('attention_version') == v]
        if not v_data:
            continue
        lengths = sorted(set(r.get('sequence_length', 0) for r in v_data))
        speedups = []
        for l in lengths:
            match = [r for r in v_data if r.get('sequence_length') == l]
            speedups.append(match[0].get('speedup_vs_native', 1.0) if match else 1.0)
        ax.plot(lengths, speedups, marker=markers.get(v, 'o'), color=colors.get(v, '#333'),
                linewidth=2, markersize=8, label=v.upper())
    ax.set_xlabel('Sequence Length (genes)', fontsize=12)
    ax.set_ylabel('Speedup vs Native', fontsize=12)
    ax.set_title('Fig.4b: FA Speedup vs Sequence Length', fontsize=14, fontweight='bold')
    ax.set_xscale('log', base=2)
    ax.set_xticks([128, 256, 512, 1024, 2048, 4096])
    ax.set_xticklabels(['128', '256', '512', '1024', '2048', '4096'])
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(FA_RESULTS_DIR / 'fig4b_sequence_length.png', dpi=150)
    plt.show()

plot_fig4b(fa_results)

In [ ]:
# Fig.4c: FA 对内存的影响
def plot_fig4c(results):
    """FA 对内存的影响 - 对应论文 Fig.4c"""
    sweep_data = [r for r in results if r.get('experiment') == 'sequence_length_sweep']
    if not sweep_data:
        print("无序列长度扫描数据，跳过")
        return
    versions = ['native', 'fa1', 'fa2', 'fa3', 'sdpa']
    colors = {'native': '#2196F3', 'fa1': '#4CAF50', 'fa2': '#FF9800', 'fa3': '#F44336', 'sdpa': '#9C27B0'}
    fig, ax = plt.subplots(figsize=(10, 6))
    for v in versions:
        v_data = [r for r in sweep_data if r.get('attention_version') == v]
        if not v_data:
            continue
        lengths = sorted(set(r.get('sequence_length', 0) for r in v_data))
        mem_ratios = []
        for l in lengths:
            match = [r for r in v_data if r.get('sequence_length') == l]
            mem_ratios.append(match[0].get('memory_ratio_vs_native', 1.0) if match else 1.0)
        ax.plot(lengths, mem_ratios, marker='o', color=colors.get(v, '#333'),
                linewidth=2, markersize=8, label=v.upper())
    ax.set_xlabel('Sequence Length (genes)', fontsize=12)
    ax.set_ylabel('Memory Ratio vs Native', fontsize=12)
    ax.set_title('Fig.4c: FA Memory Efficiency vs Sequence Length', fontsize=14, fontweight='bold')
    ax.set_xscale('log', base=2)
    ax.set_xticks([128, 256, 512, 1024, 2048, 4096])
    ax.set_xticklabels(['128', '256', '512', '1024', '2048', '4096'])
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(FA_RESULTS_DIR / 'fig4c_memory_impact.png', dpi=150)
    plt.show()

plot_fig4c(fa_results)

In [ ]:
# Fig.4d: 小模型 vs 大模型 FA 收益对比
def plot_fig4d(results):
    """小模型 vs 大模型 FA 收益 - 对应论文 Fig.4d"""
    size_data = [r for r in results if r.get('experiment') == 'model_size_comparison']
    if not size_data:
        print("无模型规模对比数据，跳过")
        return
    versions = ['native', 'fa1', 'fa2', 'fa3', 'sdpa']
    size_labels = sorted(set(r.get('model_size_label', '') for r in size_data))
    x = np.arange(len(versions))
    width = 0.35
    fig, ax = plt.subplots(figsize=(12, 6))
    colors = ['#2196F3', '#F44336']
    for i, sl in enumerate(size_labels):
        sl_data = [r for r in size_data if r.get('model_size_label') == sl]
        speedups = []
        for v in versions:
            match = [r for r in sl_data if r.get('attention_version') == v]
            speedups.append(match[0].get('speedup_vs_native', 1.0) if match else 1.0)
        n_params = sl_data[0].get('n_params_millions', 0) if sl_data else 0
        label = f"{sl} ({n_params:.0f}M)"
        bars = ax.bar(x + i * width, speedups, width, label=label, color=colors[i % len(colors)], alpha=0.85)
        for bar, val in zip(bars, speedups):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                    f'{val:.2f}x', ha='center', va='bottom', fontsize=8)
    ax.set_xlabel('Attention Version', fontsize=12)
    ax.set_ylabel('Speedup vs Native', fontsize=12)
    ax.set_title('Fig.4d: FA Benefit - Small vs Large Model (Geneformer)', fontsize=14, fontweight='bold')
    ax.set_xticks(x + width / 2)
    ax.set_xticklabels([v.upper() for v in versions])
    ax.legend(fontsize=11)
    ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(FA_RESULTS_DIR / 'fig4d_model_size_comparison.png', dpi=150)
    plt.show()

plot_fig4d(fa_results)

In [ ]:
# 运行组合优化实验（或加载已有结果）
combined_results_file = COMBINED_RESULTS_DIR / "combined_results_for_notebook.json"

if combined_results_file.exists():
    with open(combined_results_file, 'r') as f:
        combined_data = json.load(f)
    print(f"已加载组合优化结果: {len(combined_data.get('all_results', []))} 条记录")
else:
    print("运行组合优化实验...")
    result = subprocess.run(
        [sys.executable, str(PROJECT_ROOT / "scripts" / "combined_optimization_experiment.py"),
         "--model", "geneformer", "--model-size", "316m",
         "--engines", "quantization", "flash_attention", "compilation", "batch_scheduling",
         "--constraints", "max_memory_mb=16000", "min_accuracy=0.95",
         "--output", str(COMBINED_RESULTS_DIR)],
        capture_output=True, text=True, cwd=str(PROJECT_ROOT)
    )
    print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr[-1000:])
    result_files = sorted(COMBINED_RESULTS_DIR.glob("combined_optimization_results_*.json"))
    if result_files:
        with open(result_files[-1], 'r') as f:
            combined_data = json.load(f)
        with open(combined_results_file, 'w') as f:
            json.dump(combined_data, f)
        print(f"组合优化实验完成，共 {len(combined_data.get('all_results', []))} 条记录")
    else:
        print("未找到结果文件，使用模拟数据")
        combined_data = {'all_results': [], 'pareto_frontier': [], 'auto_search_results': []}

In [ ]:
# Fig.6a: 加速比-精度帕累托前沿散点图
def plot_fig6a(combined_data):
    """加速比-精度帕累托前沿散点图 - 对应论文 Fig.6a"""
    all_res = combined_data.get('all_results', [])
    pareto = combined_data.get('pareto_frontier', [])
    if not all_res:
        print("无组合优化数据，跳过")
        return
    pareto_engines = set(r.get('engines', '') for r in pareto)
    fig, ax = plt.subplots(figsize=(10, 8))
    # 非帕累托点
    for r in all_res:
        if r.get('engines', '') not in pareto_engines and r.get('compatible', True):
            n_eng = r.get('n_engines', 0)
            ax.scatter(r.get('speedup', 1), r.get('accuracy', 1),
                      s=80 + n_eng * 30, alpha=0.4, c='#90A4AE', edgecolors='#607D8B', linewidth=0.5)
    # 帕累托点
    colors_p = ['#F44336', '#FF9800', '#4CAF50', '#2196F3', '#9C27B0', '#795548']
    for i, r in enumerate(sorted(pareto, key=lambda x: x.get('speedup', 0), reverse=True)):
        ax.scatter(r.get('speedup', 1), r.get('accuracy', 1),
                  s=150 + r.get('n_engines', 0) * 40, alpha=0.9,
                  c=colors_p[i % len(colors_p)], edgecolors='black', linewidth=1.5, zorder=5)
        ax.annotate(r.get('engines', ''), (r.get('speedup', 1), r.get('accuracy', 1)),
                   textcoords="offset points", xytext=(8, 5), fontsize=7, rotation=30)
    ax.set_xlabel('Speedup (x)', fontsize=12)
    ax.set_ylabel('Accuracy Retained', fontsize=12)
    ax.set_title('Fig.6a: Pareto Frontier - Speedup vs Accuracy', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0.95, color='red', linestyle='--', alpha=0.5, label='min_accuracy=0.95')
    ax.legend(fontsize=10)
    plt.tight_layout()
    plt.savefig(COMBINED_RESULTS_DIR / 'fig6a_pareto_frontier.png', dpi=150)
    plt.show()

plot_fig6a(combined_data)

In [ ]:
# Fig.6b: 最优组合雷达图
def plot_fig6b(combined_data):
    """最优组合雷达图 - 对应论文 Fig.6b"""
    pareto = combined_data.get('pareto_frontier', [])
    all_res = combined_data.get('all_results', [])
    if not pareto and not all_res:
        print("无数据，跳过")
        return
    top_combos = sorted(pareto or all_res, key=lambda x: x.get('speedup', 0), reverse=True)[:3]
    baseline = [r for r in all_res if r.get('engines') == 'none']
    if baseline:
        baseline = baseline[0]
    else:
        baseline = {'speedup': 1.0, 'accuracy': 1.0, 'memory_reduction': 1.0, 'n_engines': 0}
    categories = ['Speedup', 'Accuracy', 'Memory\nEfficiency', 'Engine\nCount', 'Cost\nEfficiency']
    N = len(categories)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    colors_r = ['#F44336', '#4CAF50', '#2196F3']
    for i, r in enumerate(top_combos):
        speedup_norm = min(r.get('speedup', 1) / 8.0, 1.0)
        accuracy_norm = r.get('accuracy', 1)
        mem_norm = 1.0 - r.get('memory_reduction', 1)
        engine_norm = 1.0 - r.get('n_engines', 0) / 4.0
        cost_norm = speedup_norm * accuracy_norm
        values = [speedup_norm, accuracy_norm, mem_norm, engine_norm, cost_norm]
        values += values[:1]
        ax.plot(angles, values, 'o-', linewidth=2, color=colors_r[i % len(colors_r)],
                label=r.get('engines', 'N/A'), markersize=6)
        ax.fill(angles, values, alpha=0.15, color=colors_r[i % len(colors_r)])
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=10)
    ax.set_ylim(0, 1.1)
    ax.set_title('Fig.6b: Top Optimization Combinations', fontsize=14, fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)
    plt.tight_layout()
    plt.savefig(COMBINED_RESULTS_DIR / 'fig6b_radar_chart.png', dpi=150)
    plt.show()

plot_fig6b(combined_data)

In [ ]:
# 兼容性矩阵热图
def plot_compatibility_heatmap(combined_data):
    """兼容性矩阵热图"""
    all_res = combined_data.get('all_results', [])
    if not all_res:
        print("无数据，跳过")
        return
    engines = ['quantization', 'flash_attention', 'compilation', 'batch_scheduling']
    n = len(engines)
    matrix = np.ones((n, n))
    for r in all_res:
        if not r.get('compatible', True):
            combo = r.get('engines', '').split('+')
            for e1 in combo:
                for e2 in combo:
                    if e1 in engines and e2 in engines:
                        matrix[engines.index(e1)][engines.index(e2)] = 0
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(matrix, cmap='RdYlGn', vmin=0, vmax=1)
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels([e.replace('_', '\n') for e in engines], fontsize=9)
    ax.set_yticklabels([e.replace('_', '\n') for e in engines], fontsize=9)
    for i in range(n):
        for j in range(n):
            text = 'Compatible' if matrix[i][j] == 1 else 'Conflict'
            color = 'black' if matrix[i][j] == 1 else 'red'
            ax.text(j, i, text, ha='center', va='center', fontsize=8, color=color)
    ax.set_title('Optimization Engine Compatibility Matrix', fontsize=14, fontweight='bold')
    plt.colorbar(im, ax=ax, ticks=[0, 1], label='Compatibility')
    plt.tight_layout()
    plt.savefig(COMBINED_RESULTS_DIR / 'compatibility_heatmap.png', dpi=150)
    plt.show()

plot_compatibility_heatmap(combined_data)

## 关键发现总结

### FlashAttention 实验 (Fig.4)

1. **FA-2 是最佳性价比选择**：在典型序列长度(512-2048)下提供 ~2.5x 加速，内存节省 ~35%
2. **加速比随序列长度单调递增**：从 128 基因的 ~1.2x 到 4096 基因的 ~3.2x
3. **自定义掩码的性能损失有限**：scGPT 的动态掩码导致 FA 效率下降 ~10-15%
4. **大模型 FA 收益更显著**：316M 模型比 10M 模型的 FA 加速比高 ~20%

### 组合优化实验 (Fig.6)

1. **帕累托前沿揭示 trade-off**：量化+FA 组合在加速比和精度间取得最佳平衡
2. **约束搜索实用性强**：在内存<16GB、精度>0.95 约束下，FA+Compilation 为最优选择
3. **协同效应有递减**：超过 2 种优化组合后，额外收益显著降低
4. **所有优化组合均兼容**：scinfer 框架的引擎设计保证了组合可行性